###### =========================
#### 📊 Data Analytics - Silver to Gold Layer
###### =========================

###### This notebook transforms cleaned data from the Silver layer
###### into business-level analytics and insights.

#### Objectives:
###### - Compute global KPIs
###### - Analyze revenue and activity trends
###### - Identify top products and customers
###### - Perform RFM customer segmentation

### 📊 Global KPIs

This section provides a high-level overview of business performance using key metrics.

We calculate:
- Total Revenue
- Total Orders
- Total Customers
- Average Order Value

###### Load Silver data

In [0]:
df = spark.read.parquet("abfss://silver@adlsretaildev001.dfs.core.windows.net/online_retail/")
display(df.head(5))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,line_total,is_return,year,month
567306,21080,SET/20 RED RETROSPOT PAPER NAPKINS,72,2011-09-19T13:32:00.000Z,0.85,12752,Norway,61.2,0,2011,9
567306,21094,SET/6 RED SPOTTY PAPER PLATES,72,2011-09-19T13:32:00.000Z,0.85,12752,Norway,61.2,0,2011,9
567306,21086,SET/6 RED SPOTTY PAPER CUPS,72,2011-09-19T13:32:00.000Z,0.65,12752,Norway,46.8,0,2011,9
567306,84997D,CHILDRENS CUTLERY POLKADOT PINK,72,2011-09-19T13:32:00.000Z,3.75,12752,Norway,270.0,0,2011,9
567306,84997C,CHILDRENS CUTLERY POLKADOT BLUE,72,2011-09-19T13:32:00.000Z,3.75,12752,Norway,270.0,0,2011,9


##### 📥 Silver Data Validation

###### Observations:
- Data is successfully loaded from the Silver layer in Parquet format  
- Records appear clean, structured, and enriched with engineered features:
  - `line_total` (revenue per transaction line)
  - `is_return` (return indicator)
  - `year`, `month` (time dimensions)  

- Sample shows valid transactional data:
  - Positive quantities and prices  
  - Consistent timestamp format  
  - Presence of customer and country information  

###### Interpretation:
- Data cleaning and transformation steps were successfully applied in the Silver layer  
- Dataset is ready for analytical processing in the Gold layer  

This confirms that the Silver dataset is reliable and suitable for business analytics.

###### KPIs

In [0]:
from pyspark.sql.functions import *

kpis = df.agg(
    round(sum("line_total"),2).alias("Total_Revenue"),
    countDistinct("InvoiceNo").alias("Total_Orders"),
    countDistinct("CustomerID").alias("Total_Customers")
)

###### Add Average Order Value

In [0]:
kpis = kpis.withColumn(
    "Avg_Order_Value",
    round(kpis["Total_Revenue"] / kpis["Total_Orders"],2)
)

display(kpis)

Total_Revenue,Total_Orders,Total_Customers,Avg_Order_Value
8300065.81,22190,4372,374.05


##### 📊 Global KPIs Analysis

###### Observations:
- Total Revenue: **8.3M** → indicates strong overall sales performance  
- Total Orders: **22,190** → reflects a high transaction volume  
- Total Customers: **4,372** → shows a relatively broad customer base  
- Average Order Value: **374.05** → suggests high spending per order  

###### Interpretation:
- The business generates significant revenue with a moderate number of customers  
- The relatively high average order value indicates that customers tend to purchase multiple items or higher-value products per transaction  

###### Business Insight:
- Focus on customer retention could further increase revenue, given the strong spending behavior  
- High AOV suggests opportunities for upselling and bundled offers  

These KPIs confirm a healthy and revenue-generating e-commerce activity.

### 📈 Revenue & Activity Trends

This section analyzes how business performance evolves over time.

We focus on:
- Monthly revenue
- Number of orders
- Number of customers

In [0]:
monthly_trends = df.groupBy("year", "month") \
    .agg(
        round(sum("line_total"),2).alias("monthly_revenue"),
        countDistinct("InvoiceNo").alias("monthly_orders"),
        countDistinct("CustomerID").alias("monthly_customers")
    ) \
    .orderBy("year", "month")

display(monthly_trends)

year,month,monthly_revenue,monthly_orders,monthly_customers
2010,12,554604.02,1708,948
2011,1,475074.38,1236,783
2011,2,436546.15,1202,798
2011,3,579964.61,1619,1020
2011,4,426047.85,1384,899
2011,5,648251.08,1849,1079
2011,6,608013.16,1707,1051
2011,7,574238.48,1593,993
2011,8,616368.0,1544,980
2011,9,931440.37,2078,1302


##### 📈 Trend Analysis

###### Observations:
- Revenue shows an overall increasing trend throughout 2011  
- Significant growth from September to November, peaking at **1.13M in November**  
- December 2011 shows a sharp drop (**342K**), likely due to incomplete data or seasonality  

- Orders and customers follow a similar pattern:
  - Peak activity in **November (3086 orders, 1711 customers)**  
  - Strong growth starting from September  

###### Interpretation:
- The business experiences strong seasonality with peak performance in Q4 (especially November)  
- The alignment between revenue, orders, and customers indicates consistent demand growth rather than price-driven spikes  

###### Business Insight:
- Q4 (Sep–Nov) is the most critical period → ideal for marketing campaigns and inventory planning  
- November appears to be the highest-performing month → potential impact of holiday sales (e.g., Black Friday)  
- The drop in December should be interpreted cautiously (possible partial data)

These trends highlight clear seasonal patterns and growth opportunities for the business.

### 🏆 Top Products & Customers

This section identifies the best-performing products and most valuable customers.

We analyze:
- Top products by revenue
- Top products by quantity  
- Top customers by total spending

###### 🔹 Top Products by revenue

In [0]:
top_products = df.groupBy("StockCode", "Description") \
    .agg(round(sum("line_total"),2).alias("total_revenue")) \
    .orderBy("total_revenue", ascending=False)

display(top_products.limit(10))

StockCode,Description,total_revenue
22423,REGENCY CAKESTAND 3 TIER,132870.4
85123A,WHITE HANGING HEART T-LIGHT HOLDER,93823.85
85099B,JUMBO BAG RED RETROSPOT,83236.76
47566,PARTY BUNTING,67687.53
POST,POSTAGE,66710.24
84879,ASSORTED COLOUR BIRD ORNAMENT,56499.22
23084,RABBIT NIGHT LIGHT,51137.8
79321,CHILLI LIGHTS,45936.81
22086,PAPER CHAIN KIT 50'S CHRISTMAS,41500.48
22502,PICNIC BASKET WICKER 60 PIECES,39619.5


##### 🏆 Top Products Analysis

###### Observations:
- The top product is **REGENCY CAKESTAND 3 TIER** with revenue of **132K**, significantly ahead of others  
- Several decorative and gift-related items dominate the ranking:
  - Home decor (e.g., *T-LIGHT HOLDER*, *BIRD ORNAMENT*)
  - Seasonal or event products (e.g., *CHRISTMAS items*, *PARTY BUNTING*)  

- The presence of **POSTAGE** in top revenue indicates a notable contribution from shipping fees  

###### Interpretation:
- The business is strongly driven by **gift-oriented and decorative products**  
- High-performing items suggest a focus on **events, holidays, and home decoration**  
- Revenue concentration on a few products highlights key drivers of sales  

###### Business Insight:
- Top products should be prioritized for inventory management and promotions  
- Seasonal items (e.g., Christmas products) confirm the importance of peak periods  
- Shipping costs (POSTAGE) should be monitored as they represent a non-product revenue stream  

This analysis highlights the main revenue drivers and supports strategic product and marketing decisions.

🔹 Top Products by quantity

In [0]:
top_products_qty = df.groupBy("StockCode", "Description") \
    .agg(sum("Quantity").alias("total_quantity")) \
    .orderBy("total_quantity", ascending=False)

display(top_products_qty.limit(10))

StockCode,Description,total_quantity
84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,53215
85099B,JUMBO BAG RED RETROSPOT,45066
84879,ASSORTED COLOUR BIRD ORNAMENT,35314
85123A,WHITE HANGING HEART T-LIGHT HOLDER,34147
21212,PACK OF 72 RETROSPOT CAKE CASES,33409
22197,POPCORN HOLDER,30504
23084,RABBIT NIGHT LIGHT,27094
22492,MINI PAINT SET VINTAGE,25880
22616,PACK OF 12 LONDON TISSUES,25321
21977,PACK OF 60 PINK PAISLEY CAKE CASES,24163


##### 📦 Top Products by Quantity

###### Observations:
- The most sold product is **WORLD WAR 2 GLIDERS** with **53K units**, far ahead of others  
- Several low-price, high-volume items dominate:
  - Packaging or small items (e.g., *CAKE CASES*, *TISSUES*, *POPCORN HOLDER*)  
  - Decorative but affordable products  

- Some products appear in both rankings (quantity & revenue):
  - *JUMBO BAG RED RETROSPOT*  
  - *BIRD ORNAMENT*  
  - *T-LIGHT HOLDER*  

###### Interpretation:
- High-volume products are typically **low-cost items sold in bulk**  
- There is a clear distinction between:
  - **High-revenue products** (premium / higher price)  
  - **High-volume products** (low price / frequently purchased)  

- Products appearing in both lists are **key business drivers** (high volume + strong revenue)  

###### Business Insight:
- High-volume items are critical for maintaining steady sales flow  
- These products are ideal for promotions, bundles, or cross-selling strategies  
- Identifying products that perform well in both revenue and quantity is key for maximizing profitability  

This analysis provides a deeper understanding of product performance beyond revenue alone.

🔹 Top Customers

In [0]:
top_customers = df.groupBy("CustomerID") \
    .agg(round(sum("line_total"),2).alias("total_spent")) \
    .orderBy("total_spent", ascending=False)

display(top_customers.limit(10))

CustomerID,total_spent
14646,279489.02
18102,256438.49
17450,187482.17
14911,132572.62
12415,123725.45
14156,113384.14
17511,88125.38
16684,65892.08
13694,62653.1
15311,59419.34


##### 👥 Top Customers Analysis

###### Observations:
- The top customer (**ID 14646**) generated **~279K**, significantly higher than others  
- The top 3 customers each contributed over **180K**, showing strong individual impact  
- There is a noticeable gap between top customers and the rest of the list  

###### Interpretation:
- Revenue is partially concentrated among a small group of high-value customers  
- These customers likely represent **key accounts or highly loyal buyers**  
- The distribution suggests a typical **Pareto effect (80/20 rule)** where a minority drives a large share of revenue  

###### Business Insight:
- Top customers should be prioritized for retention (loyalty programs, personalized offers)  
- Losing a small number of these clients could significantly impact revenue  
- Opportunity to identify and nurture similar high-value customers  

This analysis highlights the strategic importance of high-spending customers in overall business performance.

#### 💎 RFM Segmentation

This section segments customers based on their purchasing behavior using RFM analysis:

- Recency: How recently a customer made a purchase  
- Frequency: How often a customer makes purchases  
- Monetary: How much a customer spends  

#### 🔢 RFM Scoring

Customers are ranked into 5 groups (quintiles) for each metric:
- Scores range from 1 to 5  
- Higher scores indicate more valuable customers  

This enables effective customer segmentation and targeting. 

###### RFM calculation

In [0]:
# Reference date = today
reference_date = current_date()

rfm = df.groupBy("CustomerID").agg(
    datediff(reference_date, max("InvoiceDate")).alias("Recency"),
    countDistinct("InvoiceNo").alias("Frequency"),
    round(sum("line_total"),2).alias("Monetary")
)

display(rfm)

CustomerID,Recency,Frequency,Monetary
17705,5274,10,1781.12
17997,5345,12,1511.3
15452,5300,3,538.73
16013,5274,54,33366.25
16333,5278,22,26626.8
15365,5296,9,1330.45
17706,5275,22,10097.37
13666,5331,3,84.65
14211,5324,13,2208.3
16412,5324,1,151.83


##### 💎 RFM Analysis

###### Observations:
- Recency values are relatively high (around 5200–5300 days), indicating that many customers have not purchased recently  
- Frequency varies significantly:
  - Some customers have very low frequency (1–3 orders)  
  - Others show strong engagement (e.g., 54 orders)  

- Monetary values also vary widely:
  - Low spenders (<200)  
  - High-value customers (>10K and even >30K)  
  - ⚠️ Several customers have **negative Monetary values**, indicating returns or refunds exceeding purchases  

###### Interpretation:
- The dataset contains a mix of:
  - Inactive or one-time customers  
  - Highly engaged and high-spending customers  

- Customers with high Frequency and Monetary (e.g., Customer 16013, 16333) represent **key business assets**  
- High Recency suggests many customers are no longer active → potential churn  
- Negative Monetary values highlight customer returns behavior, which can distort analysis  

###### Business Insight:
- Focus should be placed on re-engaging inactive customers (high Recency)  
- High Frequency & Monetary customers should be targeted for retention and loyalty programs  
- It is important to **handle or filter negative Monetary values** before further segmentation  
- Customer segmentation is essential to differentiate marketing strategies  

This analysis highlights customer diversity and the importance of targeted engagement strategies.

###### RFM scoring

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import ntile


# Define windows
window_recency = Window.orderBy("Recency")
window_frequency = Window.orderBy("Frequency")
window_monetary = Window.orderBy("Monetary")

rfm = rfm.filter(col("Monetary") > 0)

rfm = rfm.withColumn("R_score", ntile(5).over(window_recency)) \
         .withColumn("F_score", ntile(5).over(window_frequency)) \
         .withColumn("M_score", ntile(5).over(window_monetary))
    
display(rfm)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score
16446,5271,3,2.9,1,3,1
16738,5568,1,3.75,5,2,1
17956,5520,1,12.75,5,2,1
13307,5391,1,15.0,4,1,1
17763,5534,1,15.0,5,2,1
17900,5435,4,15.0,4,4,1
16093,5377,1,17.0,4,1,1
16953,5301,1,20.8,2,1,1
17986,5327,1,20.8,3,1,1
17102,5532,1,25.5,5,2,1


##### 💎 RFM Scoring (After Cleaning)

###### Observations:
- All customers now have **positive Monetary values**, ensuring meaningful segmentation  
- Most customers in this sample have **low spending and low frequency**, indicating low engagement  
- Recency scores vary, with some customers still relatively inactive (higher R_score)  

- RFM scores distribution:
  - **Low M_score (1)** → majority are low spenders  
  - **Low to medium F_score (1–3)** → limited purchase frequency  
  - **R_score varies (1–5)** → mix of recent and inactive customers  

###### Interpretation:
- After cleaning, the segmentation becomes more reliable and interpretable  
- The majority of customers appear to be **low-value or occasional buyers**  
- Only a small portion of customers are likely to be high-value segments  

###### Business Insight:
- Focus on converting low-frequency customers into repeat buyers  
- Identify and target high R_score customers for reactivation campaigns  
- RFM scoring can now be reliably used for segmentation and marketing strategies  

This cleaned RFM model provides a solid foundation for customer segmentation and decision-making.

#### 💾 Gold Layer Storage

The final dataset containing KPIs, trends, top analysis, and RFM segmentation is saved in the Gold layer.

This layer is optimized for business analytics and reporting.

In [0]:
kpis.write.mode("overwrite") \
    .parquet("abfss://gold@adlsretaildev001.dfs.core.windows.net/kpis/data/")

monthly_trends.write.mode("overwrite") \
    .parquet("abfss://gold@adlsretaildev001.dfs.core.windows.net/trends/monthly/")

top_products.write.mode("overwrite") \
    .parquet("abfss://gold@adlsretaildev001.dfs.core.windows.net/products/top_revenue/")

top_products_qty.write.mode("overwrite") \
    .parquet("abfss://gold@adlsretaildev001.dfs.core.windows.net/products/top_quantity/")

top_customers.write.mode("overwrite") \
    .parquet("abfss://gold@adlsretaildev001.dfs.core.windows.net/customers/top_customers/")

rfm.write.mode("overwrite") \
    .parquet("abfss://gold@adlsretaildev001.dfs.core.windows.net/customers/rfm/")

print("✅ Data successfully written to Gold layer")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Data successfully written to Gold layer


##### 🥇 Gold Layer (Business Analytics)

The Gold layer contains curated, business-ready datasets designed for analytics and reporting.

###### 📊 Available Datasets:

- **KPIs**
  - Global business metrics (revenue, orders, customers, AOV)

- **Trends**
  - Monthly revenue, orders, and customer evolution

- **Products**
  - Top products by revenue
  - Top products by quantity

- **Customers**
  - Top customers by total spending
  - RFM segmentation (Recency, Frequency, Monetary)

###### 🎯 Purpose:
These datasets are optimized for:
- Business intelligence dashboards  
- Data visualization tools (Power BI, Tableau)  
- Decision-making and strategic analysis  

##### ⚠️ Note:
Data in this layer is cleaned, aggregated, and structured for direct consumption.